In [4]:
import shutil
import pandas as pd
from pathlib import Path


## Balabit

In [5]:
PROJECT_ROOT = Path.cwd()

DATA_ROOT = PROJECT_ROOT / "Data" / "Balabit-dataset"

TEST_SRC = DATA_ROOT / "test_files"
PROTOCOL1_DST = DATA_ROOT / "testing_files_protocol1"
PROTOCOL2_DST = DATA_ROOT / "testing_files_protocol2"

LABEL_CSV = DATA_ROOT / "public_labels.csv"



In [6]:
labels_df = pd.read_csv(LABEL_CSV)

# 按你的定义：
# 0 = imposter
# 1 = genuine
session2label = dict(
    zip(labels_df["filename"], labels_df["is_illegal"])
)

print(f"Loaded {len(session2label)} labeled sessions")




Loaded 816 labeled sessions


In [7]:
labels_df["is_illegal"].value_counts()



is_illegal
0    411
1    405
Name: count, dtype: int64

In [8]:
def build_testing_protocol1():
    if PROTOCOL1_DST.exists():
        shutil.rmtree(PROTOCOL1_DST)
    PROTOCOL1_DST.mkdir(parents=True)

    kept = 0

    for user_dir in sorted(TEST_SRC.iterdir()):
        if not user_dir.is_dir():
            continue

        user_out = PROTOCOL1_DST / user_dir.name
        user_out.mkdir(parents=True)

        for session_path in user_dir.iterdir():
            if not session_path.is_file():
                continue

            session_name = session_path.name

            # only save sessions in public label list
            if session_name not in session2label:
                continue

            # Protocol 1: only genuine (is_illegal == 0)
            if session2label[session_name] == 0:
                shutil.copy(session_path, user_out / session_name)
                kept += 1

    print(f"[Protocol 1] Finished. Kept {kept} genuine sessions.")


In [35]:
build_testing_protocol1()

[Protocol 1] Finished. Kept 411 genuine sessions.


In [11]:
# -*- coding: utf-8 -*-
import os
import shutil
import pandas as pd
from pathlib import Path

# ============================================================
# 使用你提供的路径定义
# ============================================================
PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "Data" / "Balabit-dataset"

TEST_SRC = DATA_ROOT / "test_files"
PROTOCOL2_DST = DATA_ROOT / "testing_files_protocol2"
LABEL_CSV = DATA_ROOT / "public_labels.csv"

def build_testing_protocol2():
    # 1. 检查并读取标签文件
    if not LABEL_CSV.exists():
        print(f"[Error] 找不到标签文件: {LABEL_CSV}")
        return

    # 读取 CSV，第一列为 session 文件名，第二列为 label (0=genuine, 1=imposter)
    label_df = pd.read_csv(LABEL_CSV)
    session2label = dict(zip(label_df.iloc[:, 0].str.strip(), label_df.iloc[:, 1]))

    # 2. 初始化目标目录（清空旧数据）
    if PROTOCOL2_DST.exists():
        print(f"[Cleaning] 正在清理旧目录: {PROTOCOL2_DST}")
        shutil.rmtree(PROTOCOL2_DST)
    
    PROTOCOL2_DST.mkdir(parents=True)
    # 创建一级分类目录
    (PROTOCOL2_DST / "genuine").mkdir()
    (PROTOCOL2_DST / "imposter").mkdir()

    genuine_cnt = 0
    imposter_cnt = 0

    print(f"--- 正在按照 Protocol 2 重新组织数据 ---")

    # 3. 遍历原始测试文件夹
    # TEST_SRC 下级为 user7, user9 等目录
    for user_dir in sorted(TEST_SRC.iterdir()):
        if not user_dir.is_dir():
            continue

        user_name = user_dir.name
        
        # 遍历用户目录下的 session 文件（无后缀）
        for session_path in user_dir.iterdir():
            if not session_path.is_file():
                continue

            session_name = session_path.name

            # 检查该 session 是否在公开标签列表中
            if session_name not in session2label:
                continue

            label = session2label[session_name]
            
            # 确定分类：0 -> genuine, 1 -> imposter
            category = "genuine" if label == 0 else "imposter"
            
            # 目标结构: testing_files_protocol2 / {category} / {user_name} / {session}
            dest_user_path = PROTOCOL2_DST / category / user_name
            dest_user_path.mkdir(parents=True, exist_ok=True)

            # 执行拷贝
            shutil.copy(session_path, dest_user_path / session_name)

            if label == 0:
                genuine_cnt += 1
            else:
                imposter_cnt += 1

    print("-" * 40)
    print(f"[Protocol 2] 处理完成！")
    print(f"  路径: {PROTOCOL2_DST}")
    print(f"  Genuine 样本总数: {genuine_cnt}")
    print(f"  Imposter 样本总数: {imposter_cnt}")
    print("-" * 40)

if __name__ == "__main__":
    build_testing_protocol2()

--- 正在按照 Protocol 2 重新组织数据 ---
----------------------------------------
[Protocol 2] 处理完成！
  路径: c:\Users\lijun\Mouse_Dynamics\Data\Balabit-dataset\testing_files_protocol2
  Genuine 样本总数: 411
  Imposter 样本总数: 405
----------------------------------------


## DFL

In [5]:
import os
import shutil

SOURCE_ROOT = "Data/DFL-dataset_raw"
TARGET_ROOT = "Data/DFL-dataset"

MAX_SIZE_MB = 60
MAX_SIZE_BYTES = MAX_SIZE_MB * 1024 * 1024

# 只分段这些（GitHub 报错文件）
FILES_TO_SEGMENT = {
    "User19/2018_06_13__03_21_18.CSV",
    "User5/2018_05_27__21_43_59.CSV",
    "User4/2018_06_03__00_49_21.CSV",
    "User18/2018_08_30__20_31_55.CSV",
    "User19/2018_06_18__23_04_30.CSV",
    "User4/2018_05_26__12_50_48.CSV",
    "User7/2018_05_26__10_37_29.CSV",
}


def ensure_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)


def split_file(src_path, dst_dir):
    base_name = os.path.splitext(os.path.basename(src_path))[0]
    part_index = 0
    current_size = 0
    current_lines = []

    print(f"Segmenting: {src_path}")

    with open(src_path, "r", encoding="utf-8") as f:
        header = f.readline()

        for line in f:
            line_size = len(line.encode("utf-8"))

            if current_size + line_size > MAX_SIZE_BYTES:
                part_filename = f"{base_name}_part{part_index}.CSV"
                part_path = os.path.join(dst_dir, part_filename)

                with open(part_path, "w", encoding="utf-8") as out:
                    out.write(header)
                    out.writelines(current_lines)

                part_index += 1
                current_lines = []
                current_size = 0

            current_lines.append(line)
            current_size += line_size

        if current_lines:
            part_filename = f"{base_name}_part{part_index}.CSV"
            part_path = os.path.join(dst_dir, part_filename)

            with open(part_path, "w", encoding="utf-8") as out:
                out.write(header)
                out.writelines(current_lines)


def main():
    for user in os.listdir(SOURCE_ROOT):
        user_src = os.path.join(SOURCE_ROOT, user)
        user_dst = os.path.join(TARGET_ROOT, user)

        if not os.path.isdir(user_src):
            continue

        ensure_dir(user_dst)

        for file in os.listdir(user_src):
            if not file.lower().endswith(".csv"):
                continue

            relative_path = f"{user}/{file}"
            src_path = os.path.join(user_src, file)

            if relative_path in FILES_TO_SEGMENT:
                # 分段替换
                split_file(src_path, user_dst)
            else:
                # 原样复制
                shutil.copy2(src_path, os.path.join(user_dst, file))


if __name__ == "__main__":
    main()

Segmenting: Data/DFL-dataset_raw\User18\2018_08_30__20_31_55.CSV
Segmenting: Data/DFL-dataset_raw\User19\2018_06_13__03_21_18.CSV
Segmenting: Data/DFL-dataset_raw\User19\2018_06_18__23_04_30.CSV
Segmenting: Data/DFL-dataset_raw\User4\2018_05_26__12_50_48.CSV
Segmenting: Data/DFL-dataset_raw\User4\2018_06_03__00_49_21.CSV
Segmenting: Data/DFL-dataset_raw\User5\2018_05_27__21_43_59.CSV
Segmenting: Data/DFL-dataset_raw\User7\2018_05_26__10_37_29.CSV


In [2]:
import os
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# ==========================
# PATH CONFIG
# ==========================

ROOT = Path.cwd()
DATA_ROOT = ROOT / "Data" / "DFL-dataset_raw"

TRAIN_ROOT = DATA_ROOT / "training_files"
TEST_ROOT  = DATA_ROOT / "testing_files_protocol1"

TRAIN_ROOT.mkdir(parents=True, exist_ok=True)
TEST_ROOT.mkdir(parents=True, exist_ok=True)

# ==========================
# PROCESS
# ==========================

users = sorted([u for u in DATA_ROOT.iterdir() 
                if u.is_dir() and not u.name.startswith("training") 
                and not u.name.startswith("testing")])

print(f"Found {len(users)} users.")

for user_dir in users:
    user_name = user_dir.name
    
    train_user_dir = TRAIN_ROOT / user_name
    test_user_dir  = TEST_ROOT / user_name
    
    train_user_dir.mkdir(parents=True, exist_ok=True)
    test_user_dir.mkdir(parents=True, exist_ok=True)

    csv_files = sorted(user_dir.glob("*.CSV"))
    
    print(f"\nProcessing {user_name} ({len(csv_files)} files)")
    
    for csv_file in tqdm(csv_files):
        try:
            df = pd.read_csv(csv_file)
        except Exception as e:
            print(f"Skipping {csv_file.name}, error: {e}")
            continue
        
        if len(df) < 3:
            continue
        
        split_idx = int(len(df) * 2 / 3)
        
        df_train = df.iloc[:split_idx]
        df_test  = df.iloc[split_idx:]
        
        df_train.to_csv(train_user_dir / csv_file.name, index=False)
        df_test.to_csv(test_user_dir / csv_file.name, index=False)

print("\nDone. Files are now inside DFL-dataset_raw.")

Found 21 users.

Processing User1 (31 files)


100%|██████████| 31/31 [00:04<00:00,  6.56it/s]



Processing User10 (1 files)


100%|██████████| 1/1 [00:00<00:00,  1.40it/s]



Processing User11 (46 files)


...


Processing User8 (8 files)


100%|██████████| 8/8 [00:01<00:00,  7.28it/s]



Processing User9 (1 files)


100%|██████████| 1/1 [00:00<00:00,  1.49it/s]


Done. Files are now inside DFL-dataset_raw.


## ChaoShen


In [ ]:
import os

ROOT = "."
CHAOSHEN_PATH = os.path.join(ROOT, "Data", "ChaoShen")


folders = os.listdir(CHAOSHEN_PATH)

for folder in folders:
    old_path = os.path.join(CHAOSHEN_PATH, folder)

    
    if os.path.isdir(old_path) and folder.isdigit():
        new_name = f"user{folder}"
        new_path = os.path.join(CHAOSHEN_PATH, new_name)

        print(f"Renaming {folder} → {new_name}")
        os.rename(old_path, new_path)

print("Done.")

Renaming 1 → user1
Renaming 10 → user10
Renaming 11 → user11
Renaming 12 → user12
Renaming 13 → user13
Renaming 14 → user14
Renaming 15 → user15
Renaming 16 → user16
Renaming 17 → user17
Renaming 18 → user18
Renaming 19 → user19
Renaming 2 → user2
Renaming 20 → user20
Renaming 21 → user21
Renaming 22 → user22
Renaming 23 → user23
Renaming 24 → user24
Renaming 25 → user25
Renaming 26 → user26
Renaming 27 → user27
Renaming 28 → user28
Renaming 3 → user3
Renaming 4 → user4
Renaming 5 → user5
Renaming 6 → user6
Renaming 7 → user7
Renaming 8 → user8
Renaming 9 → user9
Done.


In [3]:
import os

ROOT = "."
BASE_PATH = os.path.join(ROOT, "Data", "ChaoShen")

THRESHOLD_MB = 50
THRESHOLD_BYTES = THRESHOLD_MB * 1024 * 1024

large_files = []

for user in os.listdir(BASE_PATH):
    user_path = os.path.join(BASE_PATH, user)
    
    if os.path.isdir(user_path):
        for file in os.listdir(user_path):
            if file.endswith(".csv"):
                file_path = os.path.join(user_path, file)
                size = os.path.getsize(file_path)
                
                if size > THRESHOLD_BYTES:
                    size_mb = size / (1024 * 1024)
                    large_files.append((file_path, size_mb))

if large_files:
    print("Files larger than 100MB:\n")
    for path, size_mb in large_files:
        print(f"{path}  →  {size_mb:.2f} MB")
else:
    print("No CSV files larger than 100MB.")

Files larger than 100MB:

.\Data\ChaoShen\user15\1.csv  →  80.82 MB
.\Data\ChaoShen\user15\11.csv  →  50.03 MB
.\Data\ChaoShen\user15\2.csv  →  57.76 MB
.\Data\ChaoShen\user15\6.csv  →  52.99 MB
